# video-app lab 03 - FIRST REAL CLIP (AnimateDiff-Lightning, light + fast)
Notebook 02 (Wan2.1 1.3B) crashed the session: its text encoder alone is ~11 GB and Colab's RAM cap killed it.
AnimateDiff-Lightning uses a 123M-param CLIP encoder instead, so the whole stack is ~4 GB -> comfortable on T4.
4 steps, 16 frames, 512x512. Expect 1-3 min on T4. Apache-2.0 + CreativeML OpenRAIL-M. Zero cost.


In [ ]:
!pip -q install "diffusers>=0.33" transformers accelerate safetensors imageio imageio-ffmpeg
import torch, diffusers; print('diffusers', diffusers.__version__, '| cuda', torch.cuda.is_available())


In [ ]:
import time, torch, gc
from diffusers import AnimateDiffPipeline, MotionAdapter, EulerDiscreteScheduler
from diffusers.utils import export_to_video
from IPython.display import Video, display

BASE = 'emilianJR/epiCRealism'                 # SD1.5 base, photoreal
LIGHT = 'ByteDance/AnimateDiff-Lightning'      # 4-step distilled motion adapter
DTYPE = torch.float16

t0 = time.time()
adapter = MotionAdapter.from_pretrained(LIGHT, torch_dtype=DTYPE)
pipe = AnimateDiffPipeline.from_pretrained(BASE, motion_adapter=adapter, torch_dtype=DTYPE)
pipe.scheduler = EulerDiscreteScheduler.from_config(
    pipe.scheduler.config, timestep_spacing='trailing', beta_schedule='linear')
pipe.load_lora_weights(LIGHT, weight_name='animatediff_lightning_4step_diffusers.safetensors', adapter_name='lightning')
pipe.set_adapters(['lightning'], adapter_weights=[1.0])
pipe.enable_vae_slicing()
pipe.to('cuda')
print('loaded in', round(time.time()-t0), 's | gpu GB', round(torch.cuda.memory_allocated()/1024**3, 2))


In [ ]:
PROMPT = ('cinematic photo of a red fox trotting through fresh snow in a pine forest, '
          'morning light rays, shallow depth of field, smooth steady camera, 35mm film')
NEG = 'bad quality, worse quality, blurry, distorted, watermark, text, flicker'

torch.manual_seed(42)
t0 = time.time()
frames = pipe(prompt=PROMPT, negative_prompt=NEG, num_frames=16,
              guidance_scale=1.0, num_inference_steps=4,
              generator=torch.Generator('cpu').manual_seed(42)).frames[0]
secs = round(time.time()-t0)
path = '/content/clip_01_fox_adl.mp4'
export_to_video(frames, path, fps=8)
print('GENERATED in', secs//60, 'min', secs%60, 's ->', path)
print('peak gpu GB:', round(torch.cuda.max_memory_allocated()/1024**3, 2))
display(Video(path, embed=True, width=512))
from google.colab import files; files.download(path)


### Reading the result
- Motion but wrong content -> base model change (same prompt, different BASE).
- Good -> that is our first self-generated clip on free hardware. Record the seconds.
- OOM -> drop to 384x384 and 12 frames in the last cell only.
